# Hafta 12 — Evrişimli Sinir Ağları (CNN)

Veri: `pcb_kusur.npz` (aynı klasörde). Colab'da `VERI = "mnist"` yaparak gerçek MNIST ile de çalışır (GPU önerilir).

In [ ]:
import numpy as np, matplotlib.pyplot as plt, torch, torch.nn as nn, torch.nn.functional as F, time
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, f1_score
torch.manual_seed(0); np.random.seed(0); cihaz = "cuda" if torch.cuda.is_available() else "cpu"; print("cihaz:", cihaz)

VERI = "pcb"           # "pcb" | "mnist" (Colab, internet gerekir)
if VERI == "pcb":
    v = np.load("pcb_kusur.npz"); siniflar = list(v["siniflar"])
    Xtr, ytr, Xte, yte = v["X_train"], v["y_train"], v["X_test"], v["y_test"]
else:
    from torchvision import datasets
    tr = datasets.MNIST("./veri", train=True, download=True); te = datasets.MNIST("./veri", train=False, download=True)
    Xtr, ytr, Xte, yte = tr.data.numpy()/255., tr.targets.numpy(), te.data.numpy()/255., te.targets.numpy(); siniflar = [str(i) for i in range(10)]
Xtr_t = torch.tensor(Xtr, dtype=torch.float32)[:, None]; Xte_t = torch.tensor(Xte, dtype=torch.float32)[:, None]   # (N,1,H,W)
ytr_t = torch.tensor(ytr).long(); yte_t = torch.tensor(yte).long(); K = len(siniflar); H = Xtr.shape[1]
print(VERI, Xtr_t.shape, Xte_t.shape, "sınıflar:", siniflar)

## 1. Veriye bakalım

In [ ]:
fig, ax = plt.subplots(K if K <= 4 else 2, 8, figsize=(10, 1.4*(K if K <= 4 else 2)))
for k in range(ax.shape[0]):
    for j, i in enumerate(np.where(ytr == k)[0][:8]): ax[k, j].imshow(Xtr[i], cmap="gray", vmin=0, vmax=1); ax[k, j].axis("off")
    ax[k, 0].set_title(siniflar[k], fontsize=9, loc="left")
plt.tight_layout(); plt.show(); print("sınıf sayıları:", np.bincount(ytr))

## 2. Konvolüsyon elle (Örnek 12.1) ve klasik filtreler

In [ ]:
from scipy.signal import correlate2d
img = np.array([[1,1,1,0,0],[0,1,1,1,0],[0,0,1,1,1],[0,0,1,1,0],[0,1,1,0,0]], float); Kx = np.array([[1,0,1],[0,1,0],[1,0,1]], float)
print("scipy:"); print(correlate2d(img, Kx, mode="valid"))
print("torch:"); print(F.conv2d(torch.tensor(img)[None, None], torch.tensor(Kx)[None, None])[0, 0].numpy())

In [ ]:
im = Xtr[np.where(ytr == 1)[0][0]] if VERI == "pcb" else Xtr[0]
filtreler = {"özdeşlik": np.array([[0,0,0],[0,1,0],[0,0,0]]), "Sobel yatay": np.array([[-1,-2,-1],[0,0,0],[1,2,1]]), "Sobel dikey": np.array([[-1,0,1],[-2,0,2],[-1,0,1]]), "bulanık": np.ones((3,3))/9, "keskin": np.array([[0,-1,0],[-1,5,-1],[0,-1,0]])}
fig, ax = plt.subplots(1, 5, figsize=(13, 2.8))
for a, (ad, k_) in zip(ax, filtreler.items()): a.imshow(np.abs(correlate2d(im, k_, mode="same")), cmap="gray"); a.set_title(ad); a.axis("off")
plt.show()

**Kendi çekirdeğinizi tasarlayın:** 'kısa devre' görüntüsündeki ince dikey köprüyü vurgulayan 3×3 bir çekirdek yazıp uygulayın.

## 3. Çıktı boyutu ve parametre sayısı (Örnek 12.2)

In [ ]:
def dogruluk(model, X, y, bs=512):
    model.eval(); d = 0
    with torch.no_grad():
        for i in range(0, len(X), bs): d += (model(X[i:i+bs].to(cihaz)).argmax(1).cpu() == y[i:i+bs]).sum().item()
    return d/len(X)
def egit(model, epoch=10, lr=1e-3, bs=64, artir=None, sessiz=False):
    model = model.to(cihaz); opt = torch.optim.Adam(model.parameters(), lr=lr); dl = DataLoader(TensorDataset(Xtr_t, ytr_t), batch_size=bs, shuffle=True); g = {"tr": [], "te": []}
    for ep in range(epoch):
        model.train()
        for xb, yb in dl:
            if artir: xb = artir(xb)
            xb, yb = xb.to(cihaz), yb.to(cihaz); opt.zero_grad(); F.cross_entropy(model(xb), yb).backward(); opt.step()
        g["tr"].append(dogruluk(model, Xtr_t, ytr_t)); g["te"].append(dogruluk(model, Xte_t, yte_t))
        if not sessiz: print(f"epoch {ep+1:2d}  eğitim {g['tr'][-1]:.4f}  test {g['te'][-1]:.4f}")
    return g
def cnn_kur(bn=False, dropout=0.0):
    kat = lambda cin, cout: [nn.Conv2d(cin, cout, 3, padding=1)] + ([nn.BatchNorm2d(cout)] if bn else []) + [nn.ReLU(), nn.MaxPool2d(2)]
    return nn.Sequential(*kat(1, 16), *kat(16, 32), nn.Flatten(), nn.Dropout(dropout), nn.Linear(32*(H//4)*(H//4), 64), nn.ReLU(), nn.Linear(64, K))

In [ ]:
cnn = cnn_kur(); x = torch.zeros(1, 1, H, H)
for kat in cnn: x = kat(x); print(f"{kat.__class__.__name__:12s} -> {tuple(x.shape)}")
print("CNN parametre:", sum(p.numel() for p in cnn.parameters()))
mlp = nn.Sequential(nn.Flatten(), nn.Linear(H*H, 256), nn.ReLU(), nn.Linear(256, 64), nn.ReLU(), nn.Linear(64, K)); print("MLP parametre:", sum(p.numel() for p in mlp.parameters()))

## 4. MLP tabanı ve küçük CNN

In [ ]:
torch.manual_seed(0); g_mlp = egit(mlp, epoch=15, sessiz=True)
torch.manual_seed(0); cnn = cnn_kur(); g_cnn = egit(cnn, epoch=15, sessiz=True)
plt.plot(g_mlp["te"], "s-", label=f"MLP test ({g_mlp['te'][-1]:.3f})"); plt.plot(g_cnn["te"], "o-", label=f"CNN test ({g_cnn['te'][-1]:.3f})")
plt.plot(g_mlp["tr"], "s--", alpha=.4, label="MLP eğitim"); plt.plot(g_cnn["tr"], "o--", alpha=.4, label="CNN eğitim"); plt.xlabel("epoch"); plt.ylabel("doğruluk"); plt.legend(); plt.grid(alpha=.3); plt.show()

## 5. Öğrenilen filtreler ve özellik haritaları

In [ ]:
W = cnn[0].weight.detach().cpu().numpy()[:, 0]
fig, ax = plt.subplots(2, 8, figsize=(10, 2.6))
for i, a in enumerate(ax.ravel()): a.imshow(W[i], cmap="RdBu_r", vmin=-abs(W).max(), vmax=abs(W).max()); a.axis("off")
plt.suptitle("1. katman filtreleri"); plt.show()
i0 = np.where(yte == (2 if VERI == "pcb" else 0))[0][0]
with torch.no_grad(): fm = torch.relu(cnn[0](Xte_t[i0:i0+1].to(cihaz)))[0].cpu().numpy()
fig, ax = plt.subplots(2, 9, figsize=(11, 2.6)); ax[0, 0].imshow(Xte[i0], cmap="gray"); ax[0, 0].set_title(siniflar[yte[i0]], fontsize=8)
for i in range(16): ax[i//8, 1 + i % 8].imshow(fm[i], cmap="viridis")
for a in ax.ravel(): a.axis("off")
plt.suptitle("özellik haritaları"); plt.show()

**Soru:** Hangi harita 'kısa devre' köprüsünü (ya da rakamın kenarlarını) en çok vurguluyor? Hangi filtreler yatay, hangileri dikey kenara benziyor?

## 6. BatchNorm + dropout + veri artırma

In [ ]:
def artir(xb):
    if torch.rand(1) < 0.5: xb = xb.flip(3)                       # yatay çevir (PCB için güvenli; MNIST'te KAPAT)
    if torch.rand(1) < 0.5: xb = xb.flip(2)
    dx, dy = np.random.randint(-2, 3, 2); return torch.roll(xb, (int(dx), int(dy)), (2, 3))
torch.manual_seed(0); cnn2 = cnn_kur(bn=True, dropout=0.3); g_cnn2 = egit(cnn2, epoch=15, artir=artir if VERI == "pcb" else None, sessiz=True)
print("küçük CNN test:", round(g_cnn["te"][-1], 4), " BN+dropout+artırma:", round(g_cnn2["te"][-1], 4))

## 7. Hata analizi

In [ ]:
cnn2.eval()
with torch.no_grad(): pred = cnn2(Xte_t.to(cihaz)).argmax(1).cpu().numpy()
ConfusionMatrixDisplay.from_predictions(yte, pred, display_labels=siniflar, cmap="Blues", xticks_rotation=45); plt.show()
print("makro-F1:", round(f1_score(yte, pred, average="macro"), 4))
yanlis = np.where(pred != yte)[0][:8]
fig, ax = plt.subplots(1, max(len(yanlis), 1), figsize=(1.4*max(len(yanlis), 1), 1.8))
for a, i in zip(np.atleast_1d(ax), yanlis): a.imshow(Xte[i], cmap="gray"); a.set_title(f"g:{siniflar[yte[i]][:6]}\nt:{siniflar[pred[i]][:6]}", fontsize=7); a.axis("off")
plt.show()

## 8. Sağlamlık: gürültü ve kaydırma

In [ ]:
rng = np.random.default_rng(0)
Xg = torch.tensor(np.clip(Xte + rng.normal(0, 0.1, Xte.shape), 0, 1), dtype=torch.float32)[:, None]; Xk = torch.roll(Xte_t, (2, 2), (2, 3))
for ad, m in (("küçük CNN", cnn), ("BN+dropout+artırma", cnn2)):
    print(f"{ad:20s} temiz {dogruluk(m, Xte_t, yte_t):.3f}  gürültü {dogruluk(m, Xg, yte_t):.3f}  kaydırma {dogruluk(m, Xk, yte_t):.3f}")

## 9. 1-B CNN: rulman titreşim sinyalleri (7. hafta verisi)

Elle özellik çıkarmadan ham sinyalden sınıflandırma.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
sig = pd.read_csv("rulman_titresim.csv"); Xs = sig.drop(columns="sinif").values.astype(np.float32); ys_ = pd.Categorical(sig.sinif).codes
Xs = (Xs - Xs.mean(1, keepdims=True)) / Xs.std(1, keepdims=True)                 # her sinyali standartlaştır
Xa, Xb, ya, yb = train_test_split(Xs, ys_, test_size=0.3, random_state=0, stratify=ys_)
Xa_t, Xb_t = torch.tensor(Xa)[:, None], torch.tensor(Xb)[:, None]; ya_t, yb_t = torch.tensor(ya).long(), torch.tensor(yb).long()
cnn1d = nn.Sequential(nn.Conv1d(1, 16, 32, stride=2), nn.BatchNorm1d(16), nn.ReLU(), nn.MaxPool1d(4),
                      nn.Conv1d(16, 32, 16), nn.BatchNorm1d(32), nn.ReLU(), nn.MaxPool1d(4),
                      nn.Conv1d(32, 64, 8), nn.BatchNorm1d(64), nn.ReLU(), nn.AdaptiveAvgPool1d(1), nn.Flatten(), nn.Linear(64, 4)).to(cihaz)
# Not: küçük veride (336 sinyal) BatchNorm olmadan 1-B CNN çok yavaş öğrenir — deneyin!
opt = torch.optim.Adam(cnn1d.parameters(), lr=1e-3); dl = DataLoader(TensorDataset(Xa_t, ya_t), batch_size=32, shuffle=True)
for ep in range(40):
    cnn1d.train()
    for xb, yb_ in dl: xb, yb_ = xb.to(cihaz), yb_.to(cihaz); opt.zero_grad(); F.cross_entropy(cnn1d(xb), yb_).backward(); opt.step()
    if ep % 10 == 9: print(f"epoch {ep+1}: test doğruluğu {dogruluk(cnn1d, Xb_t, yb_t):.3f}")
print("Karşılaştırma — 7. hafta elle özellik + orman: ~0.84; + zarf özellikleri: ~0.95")

## 10. Alıştırmalar

**Alıştırma 1.** 64×64 girdi için k = 5, p = 0, s = 2 ve p = 2 durumlarında çıktı boyutunu formülle hesaplayın; `nn.Conv2d` ile doğrulayın. 3×224×224 girdide 64 adet 7×7 filtrenin parametre sayısını `sum(p.numel())` ile doğrulayın.

In [ ]:
# Alıştırma 1

**Alıştırma 2.** `cnn_kur`'a üçüncü bir konvolüsyon bloğu (32→64) ekleyin ve ilk tam bağlı katmanı `nn.AdaptiveAvgPool2d(1)` (global average pooling) ile değiştirin. Parametre sayısı ve test doğruluğu nasıl değişti?

In [ ]:
# Alıştırma 2

**Alıştırma 3.** Kısa devre görüntülerinde köprü konumunu bulan bir 'ısı haritası' üretin: eğitilmiş CNN'de kısa devre sınıfının logitinin girdiye göre gradyanını alın (`x.requires_grad_(True)`; saliency map) ve |gradyan|'ı görüntü üzerine çizin.

In [ ]:
# Alıştırma 3

**Alıştırma 4.** Eğitim verisinin yalnızca %10'u ile CNN ve MLP'yi eğitin. Hangisi az veriye daha dayanıklı? Artırma bu durumda ne kadar yardımcı oluyor?

In [ ]:
# Alıştırma 4

**Alıştırma 5 (Colab, internet).** `torchvision.models.resnet18(weights='IMAGENET1K_V1')` ile PCB'de transfer öğrenme: görüntüleri 3 kanala kopyalayıp 64×64'e büyütün, konvolüsyonları dondurup yalnızca `fc`'yi 5 epoch eğitin. Sıfırdan CNN ile karşılaştırın; ardından son bloğu (`layer4`) da açıp düşük η (1e-4) ile ince ayar yapın.

In [ ]:
# Alıştırma 5